In [ ]:
# Устанавливаем PyTorch Geometric (понадобится для следующих шагов)
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_recall_curve, accuracy_score, precision_score, recall_score)
import copy
import warnings
warnings.filterwarnings('ignore')

# 1. ЗАГРУЗКА И СПЛИТ ДАННЫХ
nodes_df = pd.read_csv('enterprise_nodes_2934.csv')
edges_df = pd.read_csv('enterprise_edges_9000.csv')

unique_nodes = nodes_df['enterprise_id'].values
y = nodes_df['bankrupt'].values

indices = np.arange(len(unique_nodes))
train_idx, temp_idx, y_train, y_temp = train_test_split(indices, y, test_size=0.3, random_state=42, stratify=y)
val_idx, test_idx, y_val, y_test = train_test_split(temp_idx, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

train_nodes = set(unique_nodes[train_idx])
val_nodes = set(unique_nodes[val_idx])

# 2. ИНДУКТИВНОЕ ИЗВЛЕЧЕНИЕ ГРАФОВЫХ ФИЧЕЙ
print("Извлекаем графовые признаки (без заглядывания в будущее)...")
G_full = nx.from_pandas_edgelist(edges_df, 'source_id', 'target_id')

G_train = G_full.subgraph(train_nodes).copy()
G_val = G_full.subgraph(train_nodes.union(val_nodes)).copy()
G_test = G_full.copy()

def get_graph_features(G, target_nodes):
    pr = nx.pagerank(G, alpha=0.85)
    dc = nx.degree_centrality(G)
    cc = nx.clustering(G)

    features = []
    for node in target_nodes:
        features.append([
            pr.get(node, 0),
            dc.get(node, 0),
            cc.get(node, 0)
        ])
    return np.array(features)

graph_features_train = get_graph_features(G_train, unique_nodes[train_idx])
graph_features_val = get_graph_features(G_val, unique_nodes[val_idx])
graph_features_test = get_graph_features(G_test, unique_nodes[test_idx])

# 3. ОБЪЕДИНЕНИЕ И ЧЕСТНАЯ НОРМАЛИЗАЦИЯ
base_features = ['revenue', 'debt_ratio', 'liquidity', 'legal_cases', 'credit_score']
X_base_raw = nodes_df[base_features].values

X_train_raw = np.hstack([X_base_raw[train_idx], graph_features_train])
X_val_raw   = np.hstack([X_base_raw[val_idx], graph_features_val])
X_test_raw  = np.hstack([X_base_raw[test_idx], graph_features_test])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

X_scaled = np.zeros((len(unique_nodes), X_train_scaled.shape[1]), dtype=float)
X_scaled[train_idx] = X_train_scaled
X_scaled[val_idx] = X_val_scaled
X_scaled[test_idx] = X_test_scaled

# ==========================================
# 4. RANDOM FOREST (Базовые + Графовые фичи)
# ==========================================
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y[train_idx])

y_val_probs_rf = rf_model.predict_proba(X_val_scaled)[:, 1]
y_test_probs_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y[val_idx], y_val_probs_rf)
best_threshold_rf = thresholds[np.argmax((2 * precisions * recalls) / (precisions + recalls + 1e-10))]
y_test_pred_rf = (y_test_probs_rf >= best_threshold_rf).astype(int)

print("\n" + "="*50)
print("=== ЧЕСТНЫЙ BASELINE (Random Forest + Граф) ===")
print("="*50)
print(f"Оптимальный порог: {best_threshold_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_score(y[test_idx], y_test_probs_rf):.4f}")
print(f"PR-AUC:    {average_precision_score(y[test_idx], y_test_probs_rf):.4f}")
print("-" * 30)
print(f"Accuracy:  {accuracy_score(y[test_idx], y_test_pred_rf):.4f}")
print(f"Precision: {precision_score(y[test_idx], y_test_pred_rf):.4f}")
print(f"Recall:    {recall_score(y[test_idx], y_test_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y[test_idx], y_test_pred_rf):.4f}")


# ==========================================
# 5. GRAPHSAGE (Трансдуктивный конкурент)
# ==========================================
node_to_idx = {node_id: i for i, node_id in enumerate(unique_nodes)}
edges_src = edges_df['source_id'].map(node_to_idx).dropna().astype(int).values
edges_dst = edges_df['target_id'].map(node_to_idx).dropna().astype(int).values
edge_index = torch.tensor(np.vstack([np.concatenate([edges_src, edges_dst]),
                                     np.concatenate([edges_dst, edges_src])]), dtype=torch.long)

data = Data(x=torch.tensor(X_scaled, dtype=torch.float), edge_index=edge_index, y=torch.tensor(y, dtype=torch.long))
data.train_mask = torch.zeros(len(y), dtype=torch.bool); data.train_mask[train_idx] = True
data.val_mask   = torch.zeros(len(y), dtype=torch.bool); data.val_mask[val_idx] = True
data.test_mask  = torch.zeros(len(y), dtype=torch.bool); data.test_mask

Извлекаем графовые признаки (без заглядывания в будущее)...

=== ЧЕСТНЫЙ BASELINE (Random Forest + Граф) ===
Оптимальный порог: 0.1100
ROC-AUC:   0.7871
PR-AUC:    0.2951
------------------------------
Accuracy:  0.6054
Precision: 0.2576
Recall:    0.9365
F1-Score:  0.4041


tensor([False, False, False,  ..., False, False, False])

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_recall_curve, accuracy_score, precision_score, recall_score
import numpy as np
import copy

print("Подготовка данных для GraphSAGE...")

# 1. Собираем граф для PyTorch Geometric
node_to_idx = {node_id: i for i, node_id in enumerate(unique_nodes)}
edges_src = edges_df['source_id'].map(node_to_idx).dropna().astype(int).values
edges_dst = edges_df['target_id'].map(node_to_idx).dropna().astype(int).values
edge_index = torch.tensor(np.vstack([np.concatenate([edges_src, edges_dst]),
                                     np.concatenate([edges_dst, edges_src])]), dtype=torch.long)

data = Data(x=torch.tensor(X_scaled, dtype=torch.float), edge_index=edge_index, y=torch.tensor(y, dtype=torch.long))
data.train_mask = torch.zeros(len(y), dtype=torch.bool); data.train_mask[train_idx] = True
data.val_mask   = torch.zeros(len(y), dtype=torch.bool); data.val_mask[val_idx] = True
data.test_mask  = torch.zeros(len(y), dtype=torch.bool); data.test_mask[test_idx] = True

# 2. Архитектура нейросети
class RobustGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(RobustGNN, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=0.4, training=self.training)
        return self.conv2(x, edge_index)

num_neg = (data.y[data.train_mask] == 0).sum().item()
num_pos = (data.y[data.train_mask] == 1).sum().item()
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([num_neg / num_pos], dtype=torch.float))

model = RobustGNN(in_channels=data.num_node_features, hidden_channels=64, out_channels=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

best_val_pr_auc = 0
best_model_weights = None

print("Обучение GraphSAGE (200 эпох), это займет около 30 секунд...")

# 3. Цикл обучения
for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(data.x, data.edge_index).squeeze()[data.train_mask], data.y[data.train_mask].float())
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(data.x, data.edge_index).squeeze()[data.val_mask]).numpy()
        val_pr_auc = average_precision_score(data.y[data.val_mask].numpy(), val_probs)
        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_model_weights = copy.deepcopy(model.state_dict())

# 4. Тестирование лучшей модели
model.load_state_dict(best_model_weights)
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index).squeeze()
    val_probs = torch.sigmoid(out[data.val_mask]).numpy()
    test_probs = torch.sigmoid(out[data.test_mask]).numpy()

y_true_test = data.y[data.test_mask].numpy()
precisions, recalls, thresholds = precision_recall_curve(data.y[data.val_mask].numpy(), val_probs)
best_thresh_gnn = thresholds[np.argmax((2 * precisions * recalls) / (precisions + recalls + 1e-10))]
test_preds_gnn = (test_probs >= best_thresh_gnn).astype(int)

print("\n" + "="*50)
print("=== ЧЕСТНЫЙ GraphSAGE (Early Stopping + Threshold) ===")
print("="*50)
print(f"Оптимальный порог: {best_thresh_gnn:.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_true_test, test_probs):.4f}")
print(f"PR-AUC:    {average_precision_score(y_true_test, test_probs):.4f}")
print("-" * 30)
print(f"Accuracy:  {accuracy_score(y_true_test, test_preds_gnn):.4f}")
print(f"Precision: {precision_score(y_true_test, test_preds_gnn):.4f}")
print(f"Recall:    {recall_score(y_true_test, test_preds_gnn):.4f}")
print(f"F1-Score:  {f1_score(y_true_test, test_preds_gnn):.4f}")
print("="*50 + "\n")

Подготовка данных для GraphSAGE...
Обучение GraphSAGE (200 эпох), это займет около 30 секунд...

=== ЧЕСТНЫЙ GraphSAGE (Early Stopping + Threshold) ===
Оптимальный порог: 0.6184
ROC-AUC:   0.7377
PR-AUC:    0.2956
------------------------------
Accuracy:  0.7460
Precision: 0.2906
Recall:    0.5397
F1-Score:  0.3778

